In [ ]:
import pandas as pd
import re

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Replace 'path/to/imdb_raw.csv' with the actual path to your file in Google Drive
file_path = "/content/drive/MyDrive/MRS/imdbRaw(in).csv"
df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset Loaded Successfully!
Rows: 1009
Columns: 16


In [ ]:
initial_rows = len(df)
duplicate_rows = df.duplicated().sum()
missing_before = df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
text_columns = [
    "Series_Title",
    "Certificate",
    "Genre",
    "Director",
    "Star1",
    "Star2",
    "Star3",
    "Star4"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

In [ ]:
genre_mapping = {
    "Sci Fi": "Sci-Fi",
    "Science Fiction": "Sci-Fi",
    "science fiction": "Sci-Fi",
    "Drama": "Drama",
    "drama": "Drama",
    "DRAMA": "Drama",
    "Bio": "Biography"
}

df.columns = df.columns.str.strip()
df["Genre"] = df["Genre"].replace(genre_mapping)

In [ ]:
import re
def clean_runtime(runtime):
    if pd.isna(runtime):
        return runtime

    runtime = str(runtime)

    numbers = re.findall(r"\d+", runtime)

    if len(numbers) > 0:
        return numbers[0] + " min"

    return runtime

df["Runtime"] = df["Runtime"].apply(clean_runtime)

In [ ]:

if "Gross" in df.columns:
    df["Gross"] = (
        df["Gross"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df["Gross"] = pd.to_numeric(df["Gross"], errors="coerce")

if "Meta_score" in df.columns:
    df["Meta_score"] = pd.to_numeric(df["Meta_score"], errors="coerce")

# Iterate through all columns to fill remaining NaNs
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)

        elif pd.api.types.is_numeric_dtype(df[col]): # Numeric columns
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)

In [ ]:
numeric_columns = [
    "Released_Year",
    "IMDB_Rating",
    "No_of_Votes",
    "Meta_score"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.to_csv("cleaned_movies.csv", index=False)

In [ ]:
missing_after = df.isnull().sum()

report = f"""
=========================================
MOVIE DATA CLEANING REPORT
=========================================

Initial Rows            : {initial_rows}

Final Rows              : {len(df)}

Duplicate Rows Removed  : {duplicate_rows}

-----------------------------------------
Missing Values Before
-----------------------------------------

{missing_before}

-----------------------------------------
Missing Values After
-----------------------------------------

{missing_after}

=========================================
Cleaning Completed Successfully
=========================================
"""

print(report)

with open("cleaning_report.txt", "w") as file:
    file.write(report)

print("Cleaned dataset saved as cleaned_movies.csv")
print("Cleaning report saved as cleaning_report.txt")


MOVIE DATA CLEANING REPORT

Initial Rows            : 1009

Final Rows              : 1008

Duplicate Rows Removed  : 0

-----------------------------------------
Missing Values Before
-----------------------------------------

Poster_Link        9
Series_Title       1
Released_Year     10
Certificate      112
Runtime           14
    Genre          9
IMDB_Rating       12
Overview          10
Meta_score       168
Director           9
Star1             10
Star2              9
Star3              9
Star4              9
No_of_Votes        9
Gross            181
dtype: int64

-----------------------------------------
Missing Values After
-----------------------------------------

Poster_Link      0
Series_Title     0
Released_Year    0
Certificate      0
Runtime          0
Genre            0
IMDB_Rating      0
Overview         0
Meta_score       0
Director         0
Star1            0
Star2            0
Star3            0
Star4            0
No_of_Votes      0
Gross            0
dtype: int6

In [ ]:
from google.colab import files
files.download('cleaned_movies.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>